# LaneIQ EDA

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

data_path = Path("../data/raw/manifests.json")

with open(data_path, "r") as f:
    records = json.load(f)

df = pd.DataFrame(records)
df.head()

,shipment_id,origin_port,destination_port,cargo_weight_kg,cargo_type,ready_datetime,deadline_datetime,preferred_mode,estimated_value_usd
0,SHP-DDCFFF04,Chennai,Jebel Ali,667.85,Electronics,2026-05-20T03:27:54.945072+00:00,2026-06-03T03:27:54.945072+00:00,sea,28462.13
1,SHP-88E0A852,Abu Dhabi,NSICT Mumbai,236.03,Pharmaceuticals,2026-05-20T03:27:54.945072+00:00,2026-05-24T03:27:54.945072+00:00,sea,1768.51
2,SHP-298181E8,Chennai,Delhi ICD,776.96,Pharmaceuticals,2026-05-20T03:27:54.945072+00:00,2026-06-01T03:27:54.945072+00:00,air,12558.93
3,SHP-5CD0B6CD,Abu Dhabi,Chennai,105.90,Machinery,2026-05-24T03:27:54.945072+00:00,2026-06-08T03:27:54.945072+00:00,sea,4052.36
4,SHP-41CFCA11,NSICT Mumbai,Jebel Ali,625.81,Auto Parts,2026-05-24T03:27:54.945072+00:00,2026-06-05T03:27:54.945072+00:00,road,22591.41


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   shipment_id          5000 non-null   object 
 1   origin_port          5000 non-null   object 
 2   destination_port     5000 non-null   object 
 3   cargo_weight_kg      5000 non-null   float64
 4   cargo_type           5000 non-null   object 
 5   ready_datetime       5000 non-null   object 
 6   deadline_datetime    5000 non-null   object 
 7   preferred_mode       5000 non-null   object 
 8   estimated_value_usd  5000 non-null   float64
dtypes: float64(2), object(7)
memory usage: 351.7+ KB


In [3]:
df["ready_datetime"] = pd.to_datetime(df["ready_datetime"])
df["deadline_datetime"] = pd.to_datetime(df["deadline_datetime"])
df["deadline_days"] = (df["deadline_datetime"] - df["ready_datetime"]).dt.total_seconds() / 86400.0

df[["cargo_weight_kg", "deadline_days"]].describe()

,cargo_weight_kg,deadline_days
count,5000.000000,5000.000000
mean,1380.564326,12.080000
std,5522.588749,5.490358
min,1.000000,3.000000
25%,149.425000,7.000000
50%,403.750000,12.000000
75%,1135.462500,17.000000
max,293621.550000,21.000000


In [4]:
fig = px.histogram(df, x="cargo_weight_kg", nbins=40, title="Cargo weight distribution (kg)")
fig.show()

In [5]:
mode_counts = df["preferred_mode"].value_counts().rename_axis("mode").reset_index(name="count")
fig = px.bar(mode_counts, x="mode", y="count", title="Preferred mode split")
fig.show()

In [6]:
df["lane"] = df["origin_port"] + " -> " + df["destination_port"]
lane_counts = df["lane"].value_counts().head(10).reset_index()
lane_counts.columns = ["lane", "count"]
fig = px.bar(lane_counts, x="count", y="lane", orientation="h", title="Top 10 lanes by volume")
fig.show()

In [7]:
fig = px.histogram(df, x="deadline_days", nbins=20, title="Deadline window (days)")
fig.show()

In [8]:
weight_quantiles = df["cargo_weight_kg"].quantile([0.5, 0.9, 0.95]).to_dict()
deadline_stats = df["deadline_days"].describe()[["min", "50%", "max"]].to_dict()
summary = {
    "weight_quantiles": weight_quantiles,
    "deadline_days": deadline_stats,
    "top_lanes": lane_counts.to_dict(orient="records")
}
summary

{'weight_quantiles': {0.5: 403.75,
  0.9: 3002.6270000000036,
  0.95: 5156.208500000001},
 'deadline_days': {'min': 3.0, '50%': 12.0, 'max': 21.0},
 'top_lanes': [{'lane': 'Abu Dhabi -> Chennai', 'count': 276},
  {'lane': 'Delhi ICD -> NSICT Mumbai', 'count': 275},
  {'lane': 'Chennai -> Delhi ICD', 'count': 267},
  {'lane': 'Jebel Ali -> Delhi ICD', 'count': 266},
  {'lane': 'Jebel Ali -> Abu Dhabi', 'count': 263},
  {'lane': 'Abu Dhabi -> NSICT Mumbai', 'count': 257},
  {'lane': 'Delhi ICD -> Abu Dhabi', 'count': 256},
  {'lane': 'Delhi ICD -> Chennai', 'count': 256},
  {'lane': 'Abu Dhabi -> Jebel Ali', 'count': 254},
  {'lane': 'Delhi ICD -> Jebel Ali', 'count': 253}]}

In [9]:
transit_days = {"sea": 14, "air": 3, "road": 2}
cost_per_kg = {"sea": 1.2, "air": 4.5, "road": 2.0}
assumptions_df = pd.DataFrame({
    "mode": list(transit_days.keys()),
    "assumed_transit_days": list(transit_days.values()),
    "cost_per_kg_usd": [cost_per_kg[m] for m in transit_days]
})
assumptions_df

,mode,assumed_transit_days,cost_per_kg_usd
0,sea,14,1.2
1,air,3,4.5
2,road,2,2.0


## Observations
- Weight distribution is right-skewed; the 95th percentile is about 5,156 kg, so capacities near 6,000 kg cover most loads.
- Deadline windows span 3 to 21 days with a median near 12 days, suggesting moderate time-window slack for feasibility.
- Preferred mode is dominated by sea, so cost coefficients should keep sea cheapest per kg.
- Top lanes by volume include Abu Dhabi -> Chennai, Delhi ICD -> NSICT Mumbai, Chennai -> Delhi ICD, Jebel Ali -> Delhi ICD, and Jebel Ali -> Abu Dhabi.
- The mode split implies frequent sea-to-air tradeoffs when deadlines are tight.
